# Alpha-Ligand — the whole pipeline, in your browser

This notebook runs the three-step **Alpha-Ligand** pipeline on Google's computers.
Nothing is installed on your own machine, and nothing here needs a terminal.

1. **Profile the pocket** — look at the active site of a protein and work out what chemistry a binder would need (charges, greasy patches, rings, hydrogen bonds).
2. **Build molecules** — assemble real 3D molecules that carry that chemistry, with RDKit.
3. **Dock them** — place each molecule inside the pocket with AutoDock Vina and score how well it fits.

**How to run it:** click the little ▶ play button at the left of each grey code cell, top to bottom, and wait for each one to finish before starting the next. The first cell takes the longest (a few minutes) because it is installing software.

---

### Please read this before you trust any number below

* A Vina score is an **empirical estimate**, not a measurement. It is not a Kd, a Ki or an IC50, and this notebook will never print one.
* Docking scores correlate only **loosely** with real binding affinity. A good score is **not** a validated binder.
* Scores within roughly **1 kcal/mol** of each other should be treated as **tied**, not ranked — that is inside Vina's own error.
* The protein is held **rigid** and all **water molecules are removed**. Real pockets flex and real water often matters.
* The output is a **shortlist to test experimentally**, not a prediction you can take to the bench as fact.

## Step 0 — install the software

This installs RDKit (chemistry), Biopython (protein structures), meeko (prepares molecules for docking) and AutoDock Vina itself.

Vina is not a Python package: it is a single program, so we download the official Linux build from its GitHub release page and put it where the scripts can find it.

Expect this to take **2–5 minutes**. Lots of text will scroll past — that is normal. You only need to run it once per session.

In [ ]:
# --- install the Python libraries -------------------------------------------
# meeko 0.8 needs scipy and gemmi installed explicitly, so they are listed here.
!pip -q install rdkit biopython "meeko>=0.6" scipy gemmi

# --- install AutoDock Vina (a standalone program, not a pip package) --------
VINA_URL = "https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.5/vina_1.2.5_linux_x86_64"
!wget -q -O /usr/local/bin/vina $VINA_URL
!chmod +x /usr/local/bin/vina

# --- check everything actually arrived --------------------------------------
import importlib, shutil, sys

ok = True
for mod in ["rdkit", "Bio", "meeko", "scipy", "gemmi"]:
    try:
        importlib.import_module(mod)
        print("OK   ", mod)
    except Exception as e:
        ok = False
        print("FAIL ", mod, "->", e)

vina_path = shutil.which("vina")
if vina_path:
    print("OK    vina at", vina_path)
    !vina --version
else:
    ok = False
    print("FAIL  vina was not found on the PATH")

print()
print("All dependencies ready." if ok else "Something failed above - re-run this cell before continuing.")

## Step 1 — get the code

This copies the Alpha-Ligand project from GitHub onto this Colab machine, so the three scripts are available to run.

If you run this cell twice you will see *"destination path already exists"* — that is harmless, it just means the code is already here.

In [ ]:
import os

REPO_DIR = "/content/alpha-ligand"

if not os.path.isdir(REPO_DIR):
    !git clone --quiet https://github.com/adamcantcodelol/alpha-ligand.git {REPO_DIR}
else:
    print("Already cloned - pulling the latest version instead.")
    !cd {REPO_DIR} && git pull --quiet

os.chdir(REPO_DIR)
print("Working inside:", os.getcwd())
print()
!ls

## Step 2 — choose your protein and its active site

Two things to fill in:

* **`PDB_CODE`** — the 4-character code of a protein structure in the [Protein Data Bank](https://www.rcsb.org/). The example `3ERT` is the oestrogen receptor with the breast-cancer drug 4-hydroxytamoxifen bound in it, which makes it a good test case: we already know what a real binder for this pocket looks like.
* **`ACTIVE_SITE`** — the residues that line the site you care about, written as `chain:number`, separated by commas. For `3ERT` these are `A:351,A:383,A:524`.

> Finding the active-site residues is the part that needs your biology knowledge, not the computer's. Look at the paper for your protein, or at the residues sitting within a few ångströms of a bound ligand in the PDB entry.

**`CUTOFF`** is how far out from those residues (in ångströms) to look when deciding what counts as "the pocket". 6.0 is a tight pocket, 8.0 a generous one.

In [ ]:
# ---- EDIT THESE THREE LINES -------------------------------------------------
PDB_CODE    = "3ERT"                 # 4-character PDB code, or a filename you uploaded
ACTIVE_SITE = "A:351,A:383,A:524"    # chain:number, comma separated
CUTOFF      = 6.0                    # angstroms around those residues
# -----------------------------------------------------------------------------

print("Protein     :", PDB_CODE)
print("Active site :", ACTIVE_SITE)
print("Cutoff      :", CUTOFF, "A")

## Step 3 — profile the pocket

This reads the structure (downloading it from RCSB automatically) and describes the pocket: how greasy it is, its net charge, how many hydrogen-bond donors and acceptors line it, how big it is, and — most importantly — a **pharmacophore**: the shopping list of chemical groups a good binder should carry.

**How to read the output:** the ranked feature list near the bottom is the key part. A positively charged wall asks for a *negatively charged* group on the ligand. An aromatic wall asks for a *ring* to stack against. A greasy wall asks for a *greasy* group. That list is what the next step builds molecules to satisfy.

In [ ]:
!python pocket_profiler.py {PDB_CODE} {ACTIVE_SITE} --cutoff {CUTOFF}

print("\n" + "=" * 70)
print("POCKET REPORT")
print("=" * 70)
print(open(f"output/{PDB_CODE.upper()}_pocket_report.txt").read())

## Step 4 — build candidate molecules

This takes the pharmacophore from step 3 and assembles molecules that carry those groups, scores them on how well their chemistry covers the pocket's shopping list, and builds the best ones in real 3D with energy-minimised conformers. The result is an `.sdf` file you can open in any molecule viewer.

**How to read the output:** the score out of 100 only says *"does this molecule carry the right chemical groups?"*. It says nothing about whether the molecule physically fits — a top scorer can be far too bulky to get into the pocket. Step 5 is what tests that.

No synthetic-accessibility check is done either, so some of these would be awkward or impossible to actually make.

In [ ]:
!python ligand_builder.py output/{PDB_CODE.upper()}_pharmacophore.json --molecules 10 --confs 10

print("\n" + "=" * 70)
print("MOLECULE REPORT")
print("=" * 70)
print(open(f"output/{PDB_CODE.upper()}_ligand_report.txt").read())

## Step 5 — dock the molecules into the pocket

Now the real test. Vina places each molecule inside the pocket in many orientations and shapes, and reports the best score it can find in kcal/mol (**more negative = Vina likes it more**).

### The control

For `3ERT` we also dock **4-hydroxytamoxifen**, the drug that actually sits in this pocket in real life. That is a sanity check: if the setup is sound, the real drug should score at least as well as anything we invented. If the control scores badly, do not trust the other numbers — the setup is wrong, not the chemistry.

The `CONTROL_SMILES` line below is that molecule written in SMILES notation. **If you changed `PDB_CODE` to a different protein**, set `CONTROL_SMILES = None` unless you know a real binder for your pocket and can paste its SMILES in.

This cell is the slow one — roughly **2–6 minutes** for ten molecules.

In [ ]:
# A known real binder for this pocket, used as a yardstick. Set to None if you
# changed the protein and do not have one.
CONTROL_SMILES = "CC/C(=C(\\c1ccc(O)cc1)c1ccc(OCCN(C)C)cc1)c1ccccc1" if PDB_CODE.upper() == "3ERT" else None
CONTROL_NAME   = "4-hydroxytamoxifen_(native)"

cmd = (
    f"python dock_ligands.py output/{PDB_CODE.upper()}_pharmacophore.json "
    f"--ligands output/{PDB_CODE.upper()}_ligands.sdf "
    f"--receptor {PDB_CODE} "
    f"--exhaustiveness 8 --poses 5"
)
if CONTROL_SMILES:
    cmd += f' --control-smiles "{CONTROL_SMILES}" --control-name "{CONTROL_NAME}"'

print("Running:", cmd, "\n")
!{cmd}

print("\n" + "=" * 70)
print("DOCKING REPORT")
print("=" * 70)
print(open(f"output/{PDB_CODE.upper()}_docking_report.txt").read())

### Reading the docking scores honestly

* **More negative is better**, but the differences only mean something when they are large. Anything inside about **1 kcal/mol** of another molecule is **tied** — Vina cannot tell them apart.
* Compare your molecules to the **control**. If the real drug is well ahead of everything you generated, your candidates are weaker binders — usually because they are small and do not fill the pocket.
* The receptor was held **rigid** and its **waters were deleted**. Both are approximations that can flip a ranking.
* These scores are **not** affinities. There is no Kd, Ki or IC50 here and you should not convert a score into one.
* What you have is a **shortlist worth testing**, and a reason to test those rather than random molecules. That is all, and it is still genuinely useful.

## Step 6 — download the results to your computer

This saves the files onto your own machine through your browser. Your browser may ask you to allow multiple downloads — say yes.

| File | What it is |
|---|---|
| `_pocket_report.txt` | the readable pocket description |
| `_pharmacophore.json` | the pocket's chemical shopping list, machine-readable |
| `_ligands.sdf` | the generated molecules in 3D (open in PyMOL, Avogadro, ChimeraX...) |
| `_ligand_report.txt` | the ranked chemistry-only table |
| `_docked.sdf` | the docked **poses** — each molecule positioned inside the pocket |
| `_docking_report.txt` | the final scores, including the control |

To see a pose, open `_docked.sdf` together with the protein structure in a viewer.

In [ ]:
from google.colab import files
import os

code = PDB_CODE.upper()
wanted = [
    f"output/{code}_pocket_report.txt",
    f"output/{code}_pharmacophore.json",
    f"output/{code}_ligands.sdf",
    f"output/{code}_ligand_report.txt",
    f"output/{code}_docked.sdf",
    f"output/{code}_docking_report.txt",
]

for path in wanted:
    if os.path.exists(path):
        print("downloading", path)
        files.download(path)
    else:
        print("skipped (not found):", path)

---

## If something goes wrong

* **A cell fails with `No such file or directory`** — you probably skipped a cell, or an earlier step failed. Run them again in order from the top.
* **`vina: not found`** — re-run the install cell (step 0). Colab sessions reset after a while and lose installed programs.
* **The runtime disconnected** — Colab gives you a free machine and takes it back when idle. Reconnect and re-run from step 0; nothing is lost except time.
* **A different protein gives an error about residues** — check your `chain:number` values exist in that structure. The chain letter matters.

The companion web page does step 1 (pocket profiling) with no notebook at all:
<https://adamcantcodelol.github.io/alpha-ligand/>